In [0]:
%sql

DESCRIBE bakery_pos_develop.silver.res_users

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW stg_user AS SELECT
    user_id,
    user_name,
    created_at,
    updated_at,
    is_updated,
    is_deleted
FROM bakery_pos_develop.silver.res_users
QUALIFY ROW_NUMBER() OVER(
    PARTITION BY user_id
    ORDER BY updated_at
) = 1

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bakery_pos_develop.gold.dim_res_users (
    user_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    user_id INT,
    user_name STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    is_updated BOOLEAN,
    is_deleted BOOLEAN,
    PRIMARY KEY (user_sk) 
) USING DELTA

In [0]:
%sql
MERGE INTO bakery_pos_develop.gold.dim_res_users AS tgt
USING stg_user AS src
ON tgt.user_id = src.user_id

WHEN MATCHED AND src.updated_at > tgt.updated_at THEN
    UPDATE SET
        tgt.user_name = src.user_name,
        tgt.created_at = src.created_at,
        tgt.updated_at = src.updated_at,
        tgt.is_updated = src.is_updated,
        tgt.is_deleted = src.is_deleted

WHEN NOT MATCHED THEN
    INSERT (
        user_id,
        user_name,
        created_at,
        updated_at,
        is_updated,
        is_deleted
    )
    VALUES (
        src.user_id,
        src.user_name,
        src.created_at,
        src.updated_at,
        src.is_updated,
        src.is_deleted
    )